# ☀️ Solar Power Output Forecasting Using Deep Learning
**Final Year B.E./B.Tech Project**

**Dataset:** Hourly Solar PV Data (Norway)

**Models:** Linear Regression · Naive Bayes · XGBoost · LSTM · Transformer (MHA+FFN) · ARIMA · SARIMA

---

## 1. Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import warnings

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.linear_model import LinearRegression
from sklearn.naive_bayes import GaussianNB

from xgboost import XGBRegressor

from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.statespace.sarimax import SARIMAX

import tensorflow as tf
from tensorflow.keras.models import Model, Sequential
from tensorflow.keras.layers import (
    Input, Dense, LSTM, Dropout,
    MultiHeadAttention, LayerNormalization,
    GlobalAveragePooling1D, Conv1D, Add
)
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.optimizers import Adam

warnings.filterwarnings('ignore')
np.random.seed(42)
tf.random.set_seed(42)

print('=' * 60)
print('  SOLAR POWER OUTPUT FORECASTING — FINAL YEAR PROJECT')
print('=' * 60)

## 2. Load Dataset

In [ ]:
print('\n[1/7] Loading dataset...')

df = pd.read_csv('Solar data.csv')

print(f'    Raw shape      : {df.shape}')
print(f'    Columns        : {list(df.columns)}')
print(f'    Dtypes         :\n{df.dtypes}')
print(f'    Missing values :\n{df.isnull().sum()}')
df.head()

## 3. Exploratory Data Analysis

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 8))
fig.suptitle('Solar PV — Exploratory Data Analysis', fontsize=14, fontweight='bold')

# Distribution of AC Power
axes[0, 0].hist(df['AC_Power'].dropna(), bins=60, color='royalblue', edgecolor='k', linewidth=0.4)
axes[0, 0].set_title('AC Power Distribution')
axes[0, 0].set_xlabel('AC Power (kW)')

# Correlation heatmap (numeric cols)
num_cols = df.select_dtypes(include='number').columns[:8]   # first 8 numeric cols
corr = df[num_cols].corr()
sns.heatmap(corr, ax=axes[0, 1], annot=True, fmt='.2f', cmap='coolwarm',
            linewidths=0.5, cbar_kws={'shrink': 0.8})
axes[0, 1].set_title('Feature Correlation')

# AC Power over time (first 500 rows)
axes[1, 0].plot(df['AC_Power'].iloc[:500].values, color='darkorange', linewidth=0.8)
axes[1, 0].set_title('AC Power — First 500 Hours')
axes[1, 0].set_xlabel('Hour Index')
axes[1, 0].set_ylabel('AC Power (kW)')

# Irradiation vs AC Power scatter
if 'Irradiation' in df.columns:
    axes[1, 1].scatter(df['Irradiation'], df['AC_Power'], alpha=0.2, s=4, color='seagreen')
    axes[1, 1].set_title('Irradiation vs AC Power')
    axes[1, 1].set_xlabel('Irradiation (W/m²)')
    axes[1, 1].set_ylabel('AC Power (kW)')

plt.tight_layout()
plt.savefig('eda_plots.png', dpi=120, bbox_inches='tight')
plt.show()

## 4. Preprocessing

In [ ]:
print('\n[2/7] Preprocessing...')

df.columns = df.columns.str.strip()

# ── DateTime handling ──
if isinstance(df.index, pd.DatetimeIndex):
    print('DateTime already set as index')
elif 'DateTime' in df.columns:
    df['DateTime'] = pd.to_datetime(df['DateTime'])
    df = df.sort_values('DateTime').set_index('DateTime')
elif 'Date' in df.columns and 'Time' in df.columns:
    df['DateTime'] = pd.to_datetime(
        df['Date'].astype(str) + ' ' +
        df['Time'].astype(str).str.zfill(2) + ':00:00'
    )
    df = df.sort_values('DateTime').set_index('DateTime')
else:
    raise ValueError('No valid DateTime information found')

df.drop(columns=['Date', 'Time', 'Country'], errors='ignore', inplace=True)

# ── Remove duplicates & enforce hourly continuity ──
df = df[~df.index.duplicated(keep='first')]
full_range = pd.date_range(start=df.index.min(), end=df.index.max(), freq='h')
df = df.reindex(full_range)
df.index.name = 'DateTime'
df = df.ffill().bfill()

# ── Clip negative sensor readings ──
if 'Irradiation' in df.columns:
    df['Irradiation'] = df['Irradiation'].clip(lower=0)
for col in ['AC_Power', 'DC_Power']:
    if col in df.columns:
        df[col] = df[col].clip(lower=0)

# ── IQR outlier capping on AC_Power ──
Q1 = df['AC_Power'].quantile(0.25)
Q3 = df['AC_Power'].quantile(0.75)
upper_bound = Q3 + 3 * (Q3 - Q1)
df['AC_Power'] = df['AC_Power'].clip(upper=upper_bound)

print(f'    Cleaned shape  : {df.shape}')
print(f'    Missing values : {df.isna().sum().sum()}')
df.head()

## 5. Feature Engineering

In [ ]:
print('\n[3/7] Engineering features...')

# ── Temporal ──
df['Hour']       = df.index.hour
df['DayOfWeek']  = df.index.dayofweek
df['Month']      = df.index.month
df['DayOfYear']  = df.index.dayofyear
df['WeekOfYear'] = df.index.isocalendar().week.astype(int)

# ── Cyclical encoding ──
df['Hour_sin']  = np.sin(2 * np.pi * df['Hour']      / 24)
df['Hour_cos']  = np.cos(2 * np.pi * df['Hour']      / 24)
df['Month_sin'] = np.sin(2 * np.pi * df['Month']     / 12)
df['Month_cos'] = np.cos(2 * np.pi * df['Month']     / 12)
df['DOY_sin']   = np.sin(2 * np.pi * df['DayOfYear'] / 365)
df['DOY_cos']   = np.cos(2 * np.pi * df['DayOfYear'] / 365)

# ── Solar-physics interaction ──
if 'Irradiation' in df.columns and 'Module_Temperature' in df.columns:
    df['Irrad_x_Temp'] = df['Irradiation'] * (1 - 0.004 * df['Module_Temperature'])

# ── Lag features ──
for lag in [1, 2, 3, 6, 12, 24]:
    df[f'AC_lag_{lag}'] = df['AC_Power'].shift(lag)

# ── Rolling mean & std ──
for window in [3, 6, 12, 24]:
    df[f'roll_mean_{window}'] = df['AC_Power'].shift(1).rolling(window).mean()
    df[f'roll_std_{window}']  = df['AC_Power'].shift(1).rolling(window).std()

# ── Exponential weighted mean ──
df['ewm_6']  = df['AC_Power'].shift(1).ewm(span=6).mean()
df['ewm_24'] = df['AC_Power'].shift(1).ewm(span=24).mean()

df.dropna(inplace=True)

print(f'    Features created : {df.shape[1]} columns')
print(f'    Final shape      : {df.shape}')

## 6. Train / Test Split (Chronological — 80 / 20)

In [ ]:
print('\n[4/7] Splitting data (80/20 chronological)...')

TARGET    = 'AC_Power'
LEAK_COLS = [c for c in ['DC_Power', 'Daily_Yield', 'Total_Yield'] if c in df.columns]
DROP_COLS = [TARGET] + LEAK_COLS

X = df.drop(columns=DROP_COLS)
y = df[TARGET]

split    = int(len(df) * 0.80)
X_train, X_test = X.iloc[:split], X.iloc[split:]
y_train, y_test = y.iloc[:split], y.iloc[split:]

print(f'    Train samples : {len(X_train)}')
print(f'    Test  samples : {len(X_test)}')
print(f'    Features      : {X.shape[1]}')

## 7. Scaling

In [ ]:
print('\n[5/7] Scaling features...')

feat_scaler   = MinMaxScaler()
target_scaler = MinMaxScaler()

X_train_sc = feat_scaler.fit_transform(X_train)
X_test_sc  = feat_scaler.transform(X_test)

y_train_sc = target_scaler.fit_transform(y_train.values.reshape(-1, 1)).ravel()
y_test_sc  = target_scaler.transform(y_test.values.reshape(-1, 1)).ravel()

print('    Scaling complete.')

## 8. Helper Functions (Sequence Builder & Metrics)

In [ ]:
TIME_STEPS = 24
N_FEATURES = X_train_sc.shape[1]

def build_sequences(X, y, steps=24):
    """Convert flat feature array → (samples, steps, features) sequences."""
    Xs, ys = [], []
    for i in range(len(X) - steps):
        Xs.append(X[i : i + steps])
        ys.append(y[i + steps])
    return np.array(Xs), np.array(ys)

X_tr_seq, y_tr_seq = build_sequences(X_train_sc, y_train_sc, TIME_STEPS)
X_te_seq, y_te_seq = build_sequences(X_test_sc,  y_test_sc,  TIME_STEPS)

# Aligned original-scale targets (used for ALL models to ensure fair comparison)
y_test_seq_orig = target_scaler.inverse_transform(
    y_te_seq.reshape(-1, 1)
).ravel()

# Flat test aligned to same window (skip first TIME_STEPS rows)
y_test_aligned = y_test.values[TIME_STEPS:]

# ── Metric helper ──
results = []

def evaluate(name, y_true, y_pred):
    mae  = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2   = r2_score(y_true, y_pred)
    mape = np.mean(np.abs((y_true - y_pred) / (y_true + 1e-8))) * 100
    results.append([name, round(mae, 4), round(rmse, 4), round(r2, 4), round(mape, 2)])
    print(f'    [{name:30s}]  MAE={mae:.4f}  RMSE={rmse:.4f}  R²={r2:.4f}  MAPE={mape:.2f}%')

print('\n[6/7] Training & evaluating models...\n')

## 9. Model 1 — Linear Regression (Baseline)

In [ ]:
print('  ── Linear Regression')
lr = LinearRegression()
lr.fit(X_train_sc, y_train)
pred_lr = lr.predict(X_test_sc)
evaluate('Linear Regression', y_test_aligned, pred_lr[TIME_STEPS:])

## 10. Model 2 — Gaussian Naive Bayes
> Regression proxy: quantise target into 50 bins, predict bin, map back to mean.

In [ ]:
print('  ── Naive Bayes')
N_BINS = 50
y_train_binned = pd.cut(y_train, bins=N_BINS, labels=False).fillna(0).astype(int)
bin_means = y_train.groupby(y_train_binned.values).mean()

nb = GaussianNB()
nb.fit(X_train_sc, y_train_binned)
pred_nb_bins = nb.predict(X_test_sc)
pred_nb = np.array([bin_means.get(b, bin_means.mean()) for b in pred_nb_bins])
evaluate('Naive Bayes', y_test_aligned, pred_nb[TIME_STEPS:])

## 11. Model 3 — XGBoost

In [ ]:
print('  ── XGBoost')
xgb = XGBRegressor(
    n_estimators=500,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric='rmse',
    verbosity=0,
    random_state=42
)
xgb.fit(
    X_train_sc, y_train,
    eval_set=[(X_test_sc, y_test)],
    early_stopping_rounds=30,   # ← placed in fit(), not constructor
    verbose=False
)
pred_xgb = xgb.predict(X_test_sc)
evaluate('XGBoost', y_test_aligned, pred_xgb[TIME_STEPS:])

# Feature importance plot
importances = pd.Series(xgb.feature_importances_, index=X.columns).sort_values(ascending=False)[:20]
plt.figure(figsize=(10, 5))
importances.plot(kind='bar', color='steelblue', edgecolor='k', linewidth=0.5)
plt.title('XGBoost — Top 20 Feature Importances')
plt.ylabel('Importance Score')
plt.tight_layout()
plt.savefig('xgb_feature_importance.png', dpi=120, bbox_inches='tight')
plt.show()

## 12. Model 4 — LSTM (Stacked, with Dropout)

In [ ]:
print('  ── LSTM (Stacked)')

callbacks = [
    EarlyStopping(patience=10, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(patience=5, factor=0.5, verbose=1)
]

lstm_model = Sequential([
    LSTM(128, return_sequences=True, input_shape=(TIME_STEPS, N_FEATURES)),
    Dropout(0.2),
    LSTM(64, return_sequences=True),
    Dropout(0.2),
    LSTM(32),
    Dense(16, activation='relu'),
    Dense(1)
], name='Stacked_LSTM')

lstm_model.compile(optimizer=Adam(1e-3), loss='mse', metrics=['mae'])
lstm_model.summary()

history_lstm = lstm_model.fit(
    X_tr_seq, y_tr_seq,
    epochs=100,
    batch_size=64,
    validation_split=0.1,
    callbacks=callbacks,
    verbose=1
)

# Training curves
plt.figure(figsize=(10, 4))
plt.plot(history_lstm.history['loss'],     label='Train Loss')
plt.plot(history_lstm.history['val_loss'], label='Val Loss')
plt.title('LSTM — Training Loss')
plt.xlabel('Epoch'); plt.ylabel('MSE'); plt.legend(); plt.grid(alpha=0.3)
plt.tight_layout(); plt.savefig('lstm_loss.png', dpi=120, bbox_inches='tight'); plt.show()

pred_lstm_sc = lstm_model.predict(X_te_seq, verbose=0).ravel()
pred_lstm    = target_scaler.inverse_transform(pred_lstm_sc.reshape(-1, 1)).ravel()
evaluate('LSTM', y_test_seq_orig, pred_lstm)

## 13. Model 5 — Transformer (Multi-Head Attention + FFN)

**Architecture:**
```
Input → Conv1D Projection (d_model)
  → N × [ Multi-Head Self-Attention → Add & Norm
           Feed-Forward Network      → Add & Norm ]
→ Global Average Pooling
→ MLP Head → Output
```

In [ ]:
print('  ── Transformer (MHA + FFN)')

def transformer_encoder_block(x, num_heads, key_dim, ff_dim, dropout_rate=0.1):
    # Multi-Head Self-Attention
    attn_out = MultiHeadAttention(
        num_heads=num_heads, key_dim=key_dim, dropout=dropout_rate
    )(x, x)
    attn_out = Dropout(dropout_rate)(attn_out)
    x = LayerNormalization(epsilon=1e-6)(Add()([x, attn_out]))
    # Position-wise FFN
    ffn = Dense(ff_dim, activation='gelu')(x)
    ffn = Dropout(dropout_rate)(ffn)
    ffn = Dense(x.shape[-1])(ffn)
    ffn = Dropout(dropout_rate)(ffn)
    x   = LayerNormalization(epsilon=1e-6)(Add()([x, ffn]))
    return x


def build_transformer(
    seq_len, n_features,
    d_model=64, num_heads=4, ff_dim=128,
    num_blocks=3, mlp_units=None, dropout=0.1
):
    if mlp_units is None:
        mlp_units = [64, 32]
    inp = Input(shape=(seq_len, n_features), name='sequence_input')
    x   = Conv1D(filters=d_model, kernel_size=1, padding='same', name='input_projection')(inp)
    for _ in range(num_blocks):
        x = transformer_encoder_block(x, num_heads, d_model // num_heads, ff_dim, dropout)
    x = GlobalAveragePooling1D(name='global_avg_pool')(x)
    for units in mlp_units:
        x = Dense(units, activation='relu')(x)
        x = Dropout(dropout)(x)
    output = Dense(1, name='forecast_output')(x)
    return Model(inputs=inp, outputs=output, name='Transformer_Forecaster')


transformer_model = build_transformer(
    seq_len=TIME_STEPS, n_features=N_FEATURES,
    d_model=64, num_heads=4, ff_dim=128,
    num_blocks=3, mlp_units=[64, 32], dropout=0.1
)
transformer_model.compile(optimizer=Adam(5e-4), loss='mse', metrics=['mae'])
transformer_model.summary()

history_tf = transformer_model.fit(
    X_tr_seq, y_tr_seq,
    epochs=100,
    batch_size=64,
    validation_split=0.1,
    callbacks=callbacks,
    verbose=1
)

# Training curves
plt.figure(figsize=(10, 4))
plt.plot(history_tf.history['loss'],     label='Train Loss')
plt.plot(history_tf.history['val_loss'], label='Val Loss')
plt.title('Transformer — Training Loss')
plt.xlabel('Epoch'); plt.ylabel('MSE'); plt.legend(); plt.grid(alpha=0.3)
plt.tight_layout(); plt.savefig('transformer_loss.png', dpi=120, bbox_inches='tight'); plt.show()

pred_tf_sc = transformer_model.predict(X_te_seq, verbose=0).ravel()
pred_tf    = target_scaler.inverse_transform(pred_tf_sc.reshape(-1, 1)).ravel()
evaluate('Transformer (MHA+FFN)', y_test_seq_orig, pred_tf)

## 14. Model 6 — ARIMA

> **Note:** ARIMA is trained on the last 2,000 points of `y_train` to avoid long runtimes on large datasets.

In [ ]:
print('  ── ARIMA (5,1,0)')
# Use last 2000 training points to limit fitting time
arima_train = y_train.iloc[-2000:] if len(y_train) > 2000 else y_train
arima_model = ARIMA(arima_train, order=(5, 1, 0)).fit()
pred_arima  = arima_model.forecast(steps=len(y_test_aligned))
evaluate('ARIMA', y_test_aligned, pred_arima.values)

## 15. Model 7 — SARIMA

> **Note:** SARIMA with seasonal period 24 (hourly). Trained on last 2,000 points to limit runtime.

In [ ]:
print('  ── SARIMA (1,1,1)(1,1,1,24)')
sarima_train = y_train.iloc[-2000:] if len(y_train) > 2000 else y_train
sarima_model = SARIMAX(
    sarima_train,
    order=(1, 1, 1),
    seasonal_order=(1, 1, 1, 24),
    enforce_stationarity=False,
    enforce_invertibility=False
).fit(disp=False)
pred_sarima = sarima_model.forecast(steps=len(y_test_aligned))
evaluate('SARIMA', y_test_aligned, pred_sarima.values)

## 16. Results Table

In [ ]:
print('\n[7/7] Final Results\n')

results_df = pd.DataFrame(
    results,
    columns=['Model', 'MAE (kW)', 'RMSE (kW)', 'R² Score', 'MAPE (%)']
).sort_values('R² Score', ascending=False).reset_index(drop=True)

results_df.index += 1   # rank from 1

print('=' * 75)
print(results_df.to_string())
print('=' * 75)

best = results_df.iloc[0]
print(f"\n🏆  BEST MODEL : {best['Model']}")
print(f"    MAE        : {best['MAE (kW)']} kW")
print(f"    RMSE       : {best['RMSE (kW)']} kW")
print(f"    R² Score   : {best['R² Score']}")
print(f"    MAPE       : {best['MAPE (%)']}%")

# Styled display in notebook
results_df.style \
    .background_gradient(cmap='RdYlGn',   subset=['R² Score']) \
    .background_gradient(cmap='RdYlGn_r', subset=['MAE (kW)', 'RMSE (kW)', 'MAPE (%)']) \
    .set_caption('Model Performance Summary — All models evaluated on the same aligned test window')

## 17. Visualisations

In [ ]:
fig, axes = plt.subplots(4, 1, figsize=(14, 20))
fig.suptitle('Solar PV AC Power — Forecast Comparison', fontsize=16, fontweight='bold')

WINDOW = 200   # last N hours to display

# ── Plot 1: Deep learning models ──
ax = axes[0]
ax.plot(y_test_seq_orig[-WINDOW:],  label='Actual',      color='black',    linewidth=1.5)
ax.plot(pred_lstm[-WINDOW:],        label='LSTM',        color='royalblue', alpha=0.85)
ax.plot(pred_tf[-WINDOW:],          label='Transformer', color='crimson',   alpha=0.85)
ax.set_title('Deep Learning Models — Last 200 Test Hours')
ax.set_ylabel('AC Power (kW)'); ax.legend(); ax.grid(alpha=0.3)

# ── Plot 2: Ensemble & classical models ──
ax = axes[1]
ax.plot(y_test_aligned[-WINDOW:],   label='Actual',  color='black',      linewidth=1.5)
ax.plot(pred_xgb[-WINDOW:],         label='XGBoost', color='seagreen',   alpha=0.85)
ax.plot(pred_lr[-WINDOW:],          label='Lin.Reg', color='slateblue',  alpha=0.85)
ax.set_title('XGBoost & Linear Regression — Last 200 Test Hours')
ax.set_ylabel('AC Power (kW)'); ax.legend(); ax.grid(alpha=0.3)

# ── Plot 3: Time-series models ──
ax = axes[2]
ax.plot(y_test_aligned[-WINDOW:],   label='Actual', color='black',      linewidth=1.5)
ax.plot(pred_arima[-WINDOW:],       label='ARIMA',  color='darkorange', alpha=0.85)
ax.plot(pred_sarima[-WINDOW:],      label='SARIMA', color='purple',     alpha=0.85)
ax.set_title('ARIMA / SARIMA — Last 200 Test Hours')
ax.set_ylabel('AC Power (kW)'); ax.legend(); ax.grid(alpha=0.3)

# ── Plot 4: R² bar chart ──
ax = axes[3]
colors = ['gold' if i == 0 else 'steelblue' for i in range(len(results_df))]
bars = ax.barh(
    results_df['Model'][::-1],
    results_df['R² Score'][::-1],
    color=colors[::-1], edgecolor='k', linewidth=0.6
)
ax.set_xlabel('R² Score (higher is better)')
ax.set_title('Model Comparison — R² Score')
ax.axvline(0, color='k', linewidth=0.8)
for bar, val in zip(bars, results_df['R² Score'][::-1]):
    ax.text(bar.get_width() + 0.005, bar.get_y() + bar.get_height() / 2,
            f'{val:.4f}', va='center', fontsize=9)
ax.grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.savefig('model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('Chart saved → model_comparison.png')

## 18. Scatter Plot — Actual vs Predicted (Best Models)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
fig.suptitle('Actual vs Predicted — Scatter', fontsize=13, fontweight='bold')

for ax, name, y_true, y_pred, color in [
    (axes[0], 'LSTM',              y_test_seq_orig, pred_lstm, 'royalblue'),
    (axes[1], 'Transformer',       y_test_seq_orig, pred_tf,   'crimson')
]:
    ax.scatter(y_true, y_pred, alpha=0.3, s=6, color=color)
    lim = [min(y_true.min(), y_pred.min()), max(y_true.max(), y_pred.max())]
    ax.plot(lim, lim, 'k--', linewidth=1, label='Perfect fit')
    ax.set_title(name); ax.set_xlabel('Actual (kW)'); ax.set_ylabel('Predicted (kW)')
    ax.legend(); ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('scatter_actual_vs_pred.png', dpi=120, bbox_inches='tight')
plt.show()

## 19. Save All Artefacts

In [ ]:
# ── Save best model (XGBoost) ──
joblib.dump(xgb,           'best_solar_model.pkl')
print('Saved: best_solar_model.pkl')

# ── Save scalers ──
joblib.dump(feat_scaler,   'feat_scaler.pkl')
joblib.dump(target_scaler, 'target_scaler.pkl')
print('Saved: feat_scaler.pkl, target_scaler.pkl')

# ── Save feature column names ──
joblib.dump(X.columns.tolist(), 'feature_columns.pkl')
print('Saved: feature_columns.pkl')

# ── Save Keras models ──
lstm_model.save('lstm_model.keras')
transformer_model.save('transformer_model.keras')
print('Saved: lstm_model.keras, transformer_model.keras')

print('\n✅  Pipeline complete. All artefacts saved.')